# Phase 4 - Model Evaluation

Notebook này đọc **kết quả thực nghiệm đã sinh bởi Spark**, không hard-code metric và không train lại model. Trước khi chạy notebook, hãy chạy `scripts/run_phase4.*`.

In [ ]:
import json
from pathlib import Path
import pandas as pd
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('phase4-notebook').getOrCreate()
report = json.loads(Path('/workspace/output/results/phase4_summary.json').read_text(encoding='utf-8'))
report['evaluation_scope']

## So sánh mô hình

Precision, Recall và F1 bên dưới đều dành riêng cho `class = 1` (Fraud). Accuracy chỉ để tham khảo vì dữ liệu mất cân bằng mạnh.

In [ ]:
comparison = pd.read_csv('/workspace/output/results/model_comparison.csv')
comparison[['model', 'imbalance_method', 'precision', 'recall', 'f1', 'pr_auc', 'accuracy']]

## Phân tích threshold

Threshold thấp thường tăng Recall nhưng cũng tăng False Positive. Không kết luận threshold tối ưu nếu chưa có chi phí nghiệp vụ của FN và FP.

In [ ]:
thresholds = pd.read_csv('/workspace/output/results/threshold_analysis.csv')
thresholds[['slug', 'threshold', 'precision', 'recall', 'f1', 'fp', 'fn']]

## Prediction đã lưu trên HDFS

Chỉ lấy một số giao dịch đã gắn cờ để trình bày. Trạng thái `NEEDS_REVIEW` không phải là khẳng định giao dịch gian lận.

In [ ]:
predictions = spark.read.parquet('hdfs:///financial/predictions/rf_undersampling')
predictions.where('prediction = 1').orderBy('fraud_probability', ascending=False).show(20, truncate=False)

## Kết luận đúng phạm vi

- Class Weight làm Recall tăng mạnh nhưng tạo thêm nhiều False Positive.
- Mô hình không xử lý imbalance có thể đạt Accuracy cao nhưng bỏ sót phần lớn Fraud.
- PR-AUC và confusion matrix phù hợp hơn Accuracy khi so sánh trong bài toán này.
- Feature importance của Random Forest không biểu diễn quan hệ nhân quả.